# Assignment 3 — FlexGen on Colab

Run OPT-1.3B baseline + disk offload + Q1–Q4 on Colab T4.

**Setup**: Runtime → Change runtime type → **GPU (T4)**

> Colab's free T4 has only 12.7 GB RAM, so FlexGen's own download + conversion of OPT-1.3B will OOM. This notebook downloads pre-converted numpy weights from the TA's Drive instead.

## 1. Check GPU and Python version

In [ ]:
!nvidia-smi | head -15
!python --version
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory // 1024**2, 'MB')
!free -m | head -2
!df -h /content | head -2

## 2. Install FlexGen + gdown

Colab's pre-installed PyTorch is usually new enough (2.x + CUDA 12+); no need to reinstall.

In [ ]:
%cd /content
![ -d FlexLLMGen ] || git clone https://github.com/FMInference/FlexLLMGen.git
%cd FlexLLMGen
!pip install -q -e .
!pip install -q --upgrade gdown
%cd ..
# Create logs/ and offload directories
!mkdir -p logs flexgen_offload

## 3. Smoke test: OPT-125M (~250 MB, downloads in < 1 minute)

OPT-125M is small enough that FlexGen's own download + conversion won't OOM. Use it to confirm FlexGen is working.

In [ ]:
!python -m flexllmgen.flex_opt \
    --model facebook/opt-125m \
    --percent 100 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload

If you see `total throughput: XXX token/s` (in the hundreds), FlexGen is running on Colab.

## 4. Download the TA's pre-converted OPT-1.3B numpy weights

Downloads from the TA's Drive and extracts to `~/opt_weights/opt-1.3b-np/`. Takes ~2–5 minutes.

In [ ]:
FILE_ID = "11iyGneldJ6M_z3FO0uR-yGQ4DL0udLA8"

import os
OPT_WEIGHTS_DIR = os.path.expanduser("~/opt_weights")
os.makedirs(OPT_WEIGHTS_DIR, exist_ok=True)

TAR_PATH = "/content/opt-1.3b-np.tar"
EXTRACTED_DIR = os.path.join(OPT_WEIGHTS_DIR, "opt-1.3b-np")

# 1. Download the tar (skip if already downloaded)
if not os.path.exists(TAR_PATH):
    print("Downloading from Google Drive...")
    !gdown "https://drive.google.com/uc?id={FILE_ID}" -O {TAR_PATH}
else:
    print(f"{TAR_PATH} already exists, skipping download.")

# 2. Verify download (size should be ~2.7 GB)
!ls -lh {TAR_PATH}

# 3. Extract to ~/opt_weights/ (creates ~/opt_weights/opt-1.3b-np/)
if not os.path.exists(os.path.join(EXTRACTED_DIR, "decoder.embed_positions.weight")):
    print(f"Extracting to {OPT_WEIGHTS_DIR}/ ...")
    !tar -xf {TAR_PATH} -C {OPT_WEIGHTS_DIR}
else:
    print(f"{EXTRACTED_DIR} already extracted, skipping.")

# 4. Verify the extracted directory
print("\n=== Extracted directory ===")
!du -sh {EXTRACTED_DIR}
!ls {EXTRACTED_DIR} | head -5
print(f"Total files: $(ls {EXTRACTED_DIR} | wc -l)")

**Checkpoint**: you should now see `~/opt_weights/opt-1.3b-np/` (~2.7 GB) containing ~390 weight files.

## Q1. (30%) Progressive Offload Sweep

With `--gpu-batch-size 4 --prompt-len 128 --gen-len 16` fixed, run 5 weight distributions:

| Setting | `--percent` | Weight Distribution |
|---|---|---|
| (1) | `100 0 100 0 100 0` | 100% GPU |
| (2) | `50 50 100 0 100 0` | 50% GPU + 50% CPU |
| (3) | `50 0 100 0 100 0` | 50% GPU + 50% Disk |
| (4) | `0 50 100 0 100 0` | 50% CPU + 50% Disk |
| (5) | `0 0 100 0 100 0` | 100% Disk |

### Q1 dp(1): 100% GPU baseline

In [ ]:
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 100 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    2>&1 | tee logs/q1_1.log

### Q1 dp(2): 50% GPU + 50% CPU

In [ ]:
# Mirror dp(1), changing --percent to 50 50 100 0 100 0 and tee'ing to logs/q1_2.log


### Q1 dp(3): 50% GPU + 50% Disk

In [ ]:
# Mirror dp(1), changing --percent to 50 0 100 0 100 0 and tee'ing to logs/q1_3.log


### Q1 dp(4): 50% CPU + 50% Disk

In [ ]:
# Mirror dp(1), changing --percent to 0 50 100 0 100 0 and tee'ing to logs/q1_4.log


### Q1 dp(5): 100% Disk

In [ ]:
# Mirror dp(1), changing --percent to 0 0 100 0 100 0 and tee'ing to logs/q1_5.log


## Q2. (20%) Batch Size and I/O Amortization

With 100% disk offload (Q1 dp(5)) fixed, sweep `--gpu-batch-size = 1, 4, 16`.

> `--gpu-batch-size N` means FlexGen processes N independent prompts at once. The prompt content doesn't matter for this assignment; only the throughput numbers do.

### Q2 batch=1

In [ ]:
# Mirror Q1 dp(5), adding --gpu-batch-size 1 and tee'ing to logs/q2_b1.log


### Q2 batch=4 (reuse Q1 dp(5) data — no need to rerun)

### Q2 batch=16

In [ ]:
# Mirror Q1 dp(5), adding --gpu-batch-size 16 and tee'ing to logs/q2_b16.log


## Q3. (20%) Weight Compression

The `--compress-weight` flag quantizes weights from FP16 (16-bit) to 4-bit (4× compression ratio); they are dynamically de-quantized back to FP16 during loading to feed the GPU. Compare two scenarios with this flag enabled: (α) 100% GPU baseline, (β) 100% Disk offload.

Use Q1 dp(1) and dp(5) as the no-compression reference.

### Q3 (α): 100% GPU + compress

In [ ]:
# Mirror Q1 dp(1), adding --compress-weight at the end and tee'ing to logs/q3_alpha.log


### Q3 (β): 100% Disk + compress

In [ ]:
# Mirror Q1 dp(5), adding --compress-weight at the end and tee'ing to logs/q3_beta.log


## Q4. (30%) I/O Behavior and Bottleneck Analysis under Disk Offload

For 100% Disk Offload (Q1 dp(5)), do two things: **Part 1** uses `iostat` to sample disk behavior; **Part 2** uses FlexGen's `--debug-mode breakdown` to measure stage timings.

### Install sysstat (Colab does not pre-install iostat)

In [ ]:
!apt-get install -y -qq sysstat
!iostat -V | head -1

### Q4 Part 1: Sample disk I/O

Colab does not allow multiple terminals, so the cell below uses `%%bash` to run `iostat` / `free` in the background, FlexGen in the foreground, then kills the samplers.

In [ ]:
%%bash
# 1. Sample iostat and free in the background, remember the PIDs
iostat -x 1 > logs/q4_1_io_iostat.log &
IOSTAT_PID=$!
free -m -s 1 > logs/q4_1_io_free.log &
FREE_PID=$!

# 2. Run disk offload in the foreground (same setup as Q1 dp(5))
python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 0 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload 2>&1 | tee logs/q4_1_run.log

# 3. After FlexGen finishes, wait a few seconds for iostat to flush, then kill the samplers
sleep 5
kill -TERM $IOSTAT_PID $FREE_PID 2>/dev/null
wait 2>/dev/null
echo "Done. logs/q4_1_io_iostat.log $(wc -l < logs/q4_1_io_iostat.log) lines"

### Extract the 4 columns from the iostat log

Colab's disk name is `sda`. The awk below is ready to run.

In [ ]:
%%bash
awk '$1=="sda" {
    if ($2+0 > r) r=$2+0          # r/s peak
    if ($3+0 > k) k=$3+0          # rkB/s peak
    if ($7+0 > sz) sz=$7+0        # rareq-sz peak (max across samples)
    if ($NF+0 > u) u=$NF+0        # %util peak
} END {
    printf "r/s peak       = %s\n", r
    printf "rkB/s peak     = %s\n", k
    printf "rareq-sz peak  = %.2f\n", sz
    printf "%%util peak     = %s\n", u
}' logs/q4_1_io_iostat.log

### Q4 Part 2: Run `--debug-mode breakdown` for both scenarios

> `--debug-mode breakdown` only runs 20 batches to collect timing; **do not compare these throughput numbers to Q1**. Look only at `load_weight` and `compute_layer_decoding`. The output unit is **seconds**; values can differ by several orders of magnitude across scenarios, so keep at least 6 decimal places.

### Part 2 baseline (100% GPU)

In [ ]:
# Mirror Q1 dp(1), adding --debug-mode breakdown and tee'ing to logs/q4_2_baseline.log


### Part 2 disk offload (100% Disk)

In [ ]:
# Mirror Q1 dp(5), adding --debug-mode breakdown and tee'ing to logs/q4_2_disk.log
